# Day 5 (Tue Aug 18) — Let's build GPT (1h56m) — the main event

He types EVERYTHING live from an empty notebook — so this starter is just the data + targets. Work in tandem: pause when he names a thing, build it, unpause to compare. This notebook = dev scratchpad (his gpt-dev); consolidate into gpt.py at the end (CS336 tests import a .py).

**arc:** read data → encode/decode → batches → bigram baseline (val ~2.5) → the mathematical trick → single head → multi-head → feedforward → blocks + residuals + layernorm → scale up
**targets:** bigram baseline val ~2.5 · final GPT (n_embd 384, 6 layers, 6 heads, block 256, ~10M params) **val ~1.48** on shakespeare chars
**reuse from my week:** embeddings, Linear, cross-entropy, training loop, lr decay, eval-mode discipline, param-count alarm, shape walks — all of it appears again here

In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [18]:
# input.txt already downloaded (tiny shakespeare, 1,115,394 chars) — no wget needed
with open('input.txt', 'r') as f:
    text = f.read()
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [19]:
unique_text = sorted(set(text))
vocab_size = len(unique_text)
itos= {ch: i for ch, i in enumerate(unique_text)}
stoi= {i: ch for ch, i in enumerate(unique_text)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [20]:
data = torch.tensor(encode(text), dtype=torch.long)

# Split validation, and train data
n = int(0.9 * len(data))
train_data = data[:n] 
val_data = data[n:] 

In [21]:
BATCH_SIZE = 4 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 8 # what is the maximum context length for predictions?

In [22]:
def get_batch(data):
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix]) # 
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x, y

In [23]:
xb, yb = get_batch(train_data)

print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(BATCH_SIZE): # batch dimension
    for t in range(BLOCK_SIZE): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53